# First Set Up the GPU in Kaggle 

In [1]:
!nvidia-smi # to check if gpu is loaded for current session

Tue Apr 21 10:11:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Download PreRequisits

In [2]:
%%writefile requirements.txt

--extra-index-url https://download.pytorch.org/whl/cu117
torch==2.5.1+cu117
torchvision==0.20.1+cu117
accelerate>=1.1.0
tqdm>=4.66.0
numpy>=1.24.3,<2.0.0

Writing requirements.txt


In [3]:
!pip list | grep -E "torch|torchvision|accelerate|numpy|tqdm"

accelerate                               1.12.0
numpy                                    2.0.2
pytorch-ignite                           0.5.3
pytorch-lightning                        2.6.1
torch                                    2.10.0+cu128
torchao                                  0.10.0
torchaudio                               2.10.0+cu128
torchcodec                               0.10.0+cu128
torchdata                                0.11.0
torchinfo                                1.8.0
torchmetrics                             1.9.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cu128
tqdm                                     4.67.3


In [4]:
#!pip install -r requirements.txt

# Example - ( MNIST Model with CNN )

In [5]:
%%writefile dataset-mnist.py

import argparse
import torchvision.datasets as datasets
import torchvision.transforms as transforms

parser = argparse.ArgumentParser()
parser.add_argument("--data-dir", type=str, default="../../datasets/data")
args = parser.parse_args()

print(f"[*] Downloading CIFAR-10 to {args.data_dir} ...")
datasets.MNIST(args.data_dir, train=True,  download=True, transform=transforms.ToTensor())
datasets.MNIST(args.data_dir, train=False, download=True, transform=transforms.ToTensor())
print("[✓] CIFAR-10 download complete.")

Writing dataset-mnist.py


In [6]:
!python dataset-mnist.py

[*] Downloading CIFAR-10 to ../../datasets/data ...
100%|██████████████████████████████████████| 9.91M/9.91M [00:01<00:00, 5.80MB/s]
100%|███████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 1.45MB/s]
100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 22.4MB/s]
[✓] CIFAR-10 download complete.


In [7]:
%%writefile mnist-code.py

import time 
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class MnistModel(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv2d(input_shape, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 7 * 7, output_shape),
        )

    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.classifier(x)
        return x

def train(model, train_loader, loss_fn, optimizer, epoch, device):

    start_time = time.time()

    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    for x_train, y_train in train_loader:
        x_train, y_train = x_train.to(device), y_train.to(device)

        y_pred = model(x_train)  # Forward pass
        loss = loss_fn(y_pred, y_train)  # Compute loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted_labels = y_pred.argmax(dim=1)
        correct += (predicted_labels == y_train).sum().item()
        total_samples += y_train.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total_samples

    epoch_time = time.time() - start_time

    print(f"Epoch [{epoch}] | Train-Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}% | Time: {epoch_time:.2f}s")


def test(model, test_loader, loss_fn, epoch, device):

    start_time = time.time()

    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)

            y_pred = model(x_test)
            loss = loss_fn(y_pred, y_test)
            total_loss += loss.item()

            predicted_labels = y_pred.argmax(dim=1)
            correct += (predicted_labels == y_test).sum().item()
            total_samples += y_test.size(0)

    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total_samples

    epoch_time = time.time() - start_time

    print(f"Epoch [{epoch}] | Test Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}% | Time: {epoch_time:.2f}s")

def main():

    batch_size = 32
    epochs = 10

    torch.manual_seed(42)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_dataset = datasets.MNIST("../../datasets/data", train=True, download=False, transform=transform)
    test_dataset = datasets.MNIST("../../datasets/data", train=False, download=False, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, pin_memory=True)

    model = MnistModel(input_shape=1, hidden_units=10, output_shape=10).to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(1, epochs + 1):
        train(model, train_loader, loss_fn, optimizer, epoch, device)
        test(model, test_loader, loss_fn, epoch, device)

if __name__ == "__main__":
    main()


Writing mnist-code.py


In [8]:
! python mnist-code.py

Epoch [1] | Train-Loss: 0.1585 | Acc: 95.23% | Time: 16.32s
Epoch [1] | Test Loss: 0.1736 | Accuracy: 94.28% | Time: 2.11s
Epoch [2] | Train-Loss: 0.0890 | Acc: 97.36% | Time: 15.62s
Epoch [2] | Test Loss: 0.0854 | Accuracy: 97.17% | Time: 2.13s
Epoch [3] | Train-Loss: 0.0866 | Acc: 97.42% | Time: 15.57s
Epoch [3] | Test Loss: 0.0981 | Accuracy: 97.22% | Time: 2.06s
Epoch [4] | Train-Loss: 0.0875 | Acc: 97.26% | Time: 15.35s
Epoch [4] | Test Loss: 0.1032 | Accuracy: 97.12% | Time: 2.08s
Epoch [5] | Train-Loss: 0.0878 | Acc: 97.35% | Time: 15.80s
Epoch [5] | Test Loss: 0.0869 | Accuracy: 97.57% | Time: 2.11s
Epoch [6] | Train-Loss: 0.0851 | Acc: 97.38% | Time: 15.59s
Epoch [6] | Test Loss: 0.0747 | Accuracy: 97.73% | Time: 2.10s
Epoch [7] | Train-Loss: 0.0885 | Acc: 97.30% | Time: 15.46s
Epoch [7] | Test Loss: 0.0825 | Accuracy: 97.51% | Time: 2.10s
Epoch [8] | Train-Loss: 0.0828 | Acc: 97.61% | Time: 15.43s
Epoch [8] | Test Loss: 0.1010 | Accuracy: 97.04% | Time: 2.00s
Epoch [9] | Trai

# Mnist Model with CNN - DDP

In [9]:
%%writefile mnist-code-ddp.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import argparse

class MnistModel(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv2d(input_shape, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 7 * 7, output_shape),
        )

    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.classifier(x)
        return x

def train(rank, model, train_loader, loss_fn, optimizer, epoch, device):

    start_time = time.time()

    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    for x_train, y_train in train_loader:
        x_train, y_train = x_train.to(device), y_train.to(device)

        y_pred = model(x_train)  # Forward pass
        loss = loss_fn(y_pred, y_train)  # Compute loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted_labels = y_pred.argmax(dim=1)
        correct += (predicted_labels == y_train).sum().item()
        total_samples += y_train.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total_samples

    epoch_time = time.time() - start_time

    if rank == 0:
        print(f"Epoch [{epoch}] | Train-Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}% | Time: {epoch_time:.2f}s")


def test(rank, model, test_loader, loss_fn, epoch, device):

    start_time = time.time()

    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)

            y_pred = model(x_test)
            loss = loss_fn(y_pred, y_test)
            total_loss += loss.item()

            predicted_labels = y_pred.argmax(dim=1)
            correct += (predicted_labels == y_test).sum().item()
            total_samples += y_test.size(0)

    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total_samples

    epoch_time = time.time() - start_time

    if rank == 0:
        print(f"Epoch [{epoch}] | Test Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}% | Time: {epoch_time:.2f}s")

def setup():
    dist.init_process_group(backend="nccl")

def cleanup():
    dist.destroy_process_group()

def main():
    parser = argparse.ArgumentParser(description="PyTorch DDP MNIST Example")
    parser.add_argument("--epochs", type=int, default=10, help="Number of epochs")
    parser.add_argument("--batch-size", type=int, default=32, help="Batch size per GPU")
    parser.add_argument("--save-model", action="store_true", help="Save trained model")
    parser.add_argument("--local_rank", type=int, default=0, help="Local rank (assigned by torch.distributed.launch)")
    args = parser.parse_args()

    torch.manual_seed(42)

    # Setup the process group
    setup()

    # Get the rank and world size from the environment
    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", args.local_rank))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(local_rank)

    if rank == 0:
        print(f"[INFO] Using {world_size} GPUs")

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_dataset = datasets.MNIST("../../datasets/data", train=True, download=False, transform=transform)
    test_dataset = datasets.MNIST("../../datasets/data", train=False, download=False, transform=transform)

    train_sampler = DistributedSampler(train_dataset)
    test_sampler = DistributedSampler(test_dataset, shuffle=False)

    train_loader = DataLoader(train_dataset, sampler=train_sampler, batch_size=args.batch_size // world_size, pin_memory=True)
    test_loader = DataLoader(test_dataset, sampler=test_sampler, batch_size=args.batch_size // world_size, pin_memory=True)

    model = MnistModel(input_shape=1, hidden_units=10, output_shape=10).to(device)
    model = DDP(model, device_ids=[local_rank])

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(1, args.epochs + 1):
        train_sampler.set_epoch(epoch)  # Important for proper shuffling
        train(rank, model, train_loader, loss_fn, optimizer, epoch, device)
        test(rank, model, test_loader, loss_fn, epoch, device)

    if rank == 0 and args.save_model:
        torch.save(model.module.state_dict(), "mnist_ddp_model.pth")

    cleanup()

if __name__ == "__main__":
    main()


Writing mnist-code-ddp.py


In [10]:
! torchrun \
    --nnodes=1 \
    --nproc-per-node=2 \
    mnist-code-ddp.py \
    --epochs=10 \
    --batch-size=32

W0420 23:36:16.281000 124 torch/distributed/run.py:852] 
W0420 23:36:16.281000 124 torch/distributed/run.py:852] *****************************************
W0420 23:36:16.281000 124 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0420 23:36:16.281000 124 torch/distributed/run.py:852] *****************************************
[INFO] Using 2 GPUs
Epoch [1] | Train-Loss: 0.1514 | Acc: 95.51% | Time: 12.52s
Epoch [1] | Test Loss: 0.0946 | Accuracy: 97.48% | Time: 1.30s
Epoch [2] | Train-Loss: 0.0918 | Acc: 97.25% | Time: 12.14s
Epoch [2] | Test Loss: 0.0824 | Accuracy: 97.68% | Time: 1.25s
Epoch [3] | Train-Loss: 0.0855 | Acc: 97.40% | Time: 12.16s
Epoch [3] | Test Loss: 0.0843 | Accuracy: 97.02% | Time: 1.26s
Epoch [4] | Train-Loss: 0.0866 | Acc: 97.39% | Time: 11.84s
Epoch [4] | Test Loss: 0.0773 | 

# Trying Different Example - ( Cifar with Resnet-50 )

In [1]:
%%writefile dataset-cifar.py

import argparse
import torchvision.datasets as datasets
import torchvision.transforms as transforms

parser = argparse.ArgumentParser()
parser.add_argument("--data-dir", type=str, default="../../datasets/data-cifar")
args = parser.parse_args()

print(f"[*] Downloading CIFAR-10 to {args.data_dir} ...")
datasets.CIFAR10(args.data_dir, train=True,  download=True, transform=transforms.ToTensor())
datasets.CIFAR10(args.data_dir, train=False, download=True, transform=transforms.ToTensor())
print("[✓] CIFAR-10 download complete.")

Writing dataset-cifar.py


In [2]:
!python dataset-cifar.py

[*] Downloading CIFAR-10 to ../../datasets/data-cifar ...
100%|████████████████████████████████████████| 170M/170M [00:03<00:00, 48.7MB/s]
[✓] CIFAR-10 download complete.


In [3]:
!ls ../../datasets/ # verify comman

data-cifar


# Single GPU Setup

In [28]:
%%writefile resnet-cifar.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def train(model, train_loader, loss_fn, optimizer, epoch, device):
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
            f"| Acc: {100*correct/total_samples:.2f}% "
            f"| Time: {(time.time()-start):.2f}s")


def test(model, test_loader, loss_fn, epoch, device):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)


    print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
            f"| Acc: {100*correct/total_samples:.2f}% "
            f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")  # ← new
    args = parser.parse_args()


    device = torch.device(f"cuda")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # ← use args.data_dir instead of hardcoded relative path
    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)


    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                               num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model = build_model().to(device)

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train( model, train_loader, loss_fn, optimizer, epoch, device)
        test( model, test_loader, loss_fn, epoch, device)
        scheduler.step()

    total = time.time() - total_start
    print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

if __name__ == "__main__":
    main()


Overwriting resnet-cifar.py


In [29]:
!python resnet-cifar.py --batch-size 512

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch [1] | Train Loss: 2.7543 | Acc: 11.1

## Multi GPU DDP Setup

In [30]:
%%writefile cifar-resnet-ddp.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device):
    train_sampler.set_epoch(epoch)
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(rank, model, test_loader, loss_fn, epoch, device):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")  # ← new
    args = parser.parse_args()

    dist.init_process_group(backend="nccl")
    rank       = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(local_rank)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # ← use args.data_dir instead of hardcoded relative path
    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True)

    # after world_size is known, divide batch
    batch_size_per_gpu = args.batch_size // world_size

    if rank == 0:
        print(f"[INFO] Nodes: {world_size // 2} | GPUs/node: 2 | World size: {world_size}")
        print(f"[INFO] Batch per GPU: {batch_size_per_gpu} | Effective batch: {args.batch_size}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size_per_gpu,  # ← divided
                          sampler=train_sampler, num_workers=args.num_workers,
                          pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size_per_gpu * 2,  # ← divided too
                          shuffle=False, num_workers=args.num_workers,
                          pin_memory=True)

    model = build_model().to(device)
    model = DDP(model, device_ids=[local_rank])

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device)
        test(rank, model, test_loader, loss_fn, epoch, device)
        scheduler.step()

    if rank == 0:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Overwriting cifar-resnet-ddp.py


In [31]:
! torchrun \
    --nnodes=1 \
    --nproc-per-node=2 \
    cifar-resnet-ddp.py \
    --epochs=10 \
    --batch-size=512

W0421 00:12:36.706000 704 torch/distributed/run.py:852] 
W0421 00:12:36.706000 704 torch/distributed/run.py:852] *****************************************
W0421 00:12:36.706000 704 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0421 00:12:36.706000 704 torch/distributed/run.py:852] *****************************************
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[I

# Multi GPU DDP Accelerate Setup

In [32]:
%%writefile cifar-resnet-ddp-accelerate.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from accelerate import Accelerator
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(accelerator, model, train_loader, loss_fn, optimizer, epoch):
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        accelerator.backward(loss)
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(accelerator, model, test_loader, loss_fn, epoch):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")
    args = parser.parse_args()

    torch.manual_seed(42)

    # Create an Accelerator object — handles device placement and distributed setup
    accelerator = Accelerator()

    # Divide batch size so effective batch stays constant regardless of GPU count.
    # accelerator.prepare() handles the DistributedSampler automatically,
    # but it does NOT divide the batch size — that's our responsibility.
    batch_size_per_gpu = args.batch_size // accelerator.num_processes

    if accelerator.is_main_process:
        print(f"[INFO] Num processes:    {accelerator.num_processes}")
        print(f"[INFO] Batch per GPU:    {batch_size_per_gpu} | Effective batch: {args.batch_size}")
        print(f"[INFO] Data dir:         {args.data_dir}")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_loader = DataLoader(train_dataset, batch_size=batch_size_per_gpu,
                              shuffle=True, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size_per_gpu * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model     = build_model()
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    # Accelerator wraps model, optimizer, and dataloaders — DDP is handled automatically
    model, optimizer, train_loader, test_loader = accelerator.prepare(
        model, optimizer, train_loader, test_loader
    )

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(accelerator, model, train_loader, loss_fn, optimizer, epoch)
        test(accelerator, model, test_loader, loss_fn, epoch)
        scheduler.step()

    if accelerator.is_main_process:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

if __name__ == "__main__":
    main()

Overwriting cifar-resnet-ddp-accelerate.py


In [ ]:
! accelerate launch \
    --num_processes 2 \
    cifar-resnet-ddp-accelerate.py \
    --epochs 10 \
    --batch-size 512

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[INFO] Num processes:    2
[INFO] Batch per GPU:    256 | Effective batch: 512
[INFO] Data dir:         ../../datasets/data-cifar
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slo

# Multi GPU FSDP Setup

In [5]:
%%writefile cifar-resnet-fsdp.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# FSDP packages
from torch.utils.data import DistributedSampler
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    BackwardPrefetch,
    ShardingStrategy,
    CPUOffload,
)
from torch.distributed.fsdp.wrap import (
    size_based_auto_wrap_policy,
    enable_wrap,
    wrap,
)
import functools
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device):
    train_sampler.set_epoch(epoch)
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(rank, model, test_loader, loss_fn, epoch, device):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def setup():
    dist.init_process_group(backend="nccl")


def cleanup():
    dist.destroy_process_group()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",            type=int,   default=10)
    parser.add_argument("--batch-size",        type=int,   default=512)
    parser.add_argument("--lr",                type=float, default=0.1)
    parser.add_argument("--num-workers",       type=int,   default=8)
    parser.add_argument("--data-dir",          type=str,   default="../../datasets/data-cifar")
    parser.add_argument("--sharding-strategy", type=str,   default="FULL_SHARD",
                        choices=["FULL_SHARD", "SHARD_GRAD_OP", "NO_SHARD"],
                        help="FSDP sharding strategy")
    parser.add_argument("--mixed-precision",   action="store_true", help="Use mixed precision (fp16)")
    parser.add_argument("--cpu-offload",       action="store_true", help="Offload parameters to CPU")
    args = parser.parse_args()

    torch.manual_seed(42)

    setup()

    rank       = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(local_rank)

    # Divide batch size so effective batch stays constant regardless of GPU count.
    # DistributedSampler splits the dataset, but batch_size is per-GPU — our responsibility.
    batch_size_per_gpu = args.batch_size // world_size

    if rank == 0:
        print(f"[INFO] Using {world_size} GPUs")
        print(f"[INFO] Sharding Strategy: {args.sharding_strategy}")
        print(f"[INFO] Mixed Precision:   {args.mixed_precision}")
        print(f"[INFO] CPU Offload:       {args.cpu_offload}")
        print(f"[INFO] Batch per GPU:     {batch_size_per_gpu} | Effective batch: {args.batch_size}")
        print(f"[INFO] Data dir:          {args.data_dir}")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size_per_gpu,
                              sampler=train_sampler, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size_per_gpu * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model = build_model().to(device)

    # Auto-wrap policy: shard any sub-module with >= 1M parameters
    my_auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy, min_num_params=1_000_000
    )

    # Sharding strategy
    sharding_map = {
        "FULL_SHARD":    ShardingStrategy.FULL_SHARD,
        "SHARD_GRAD_OP": ShardingStrategy.SHARD_GRAD_OP,
        "NO_SHARD":      ShardingStrategy.NO_SHARD,
    }
    sharding_strategy = sharding_map[args.sharding_strategy]

    # Mixed precision config
    mixed_precision_config = None
    if args.mixed_precision:
        mixed_precision_config = MixedPrecision(
            param_dtype=torch.float16,
            reduce_dtype=torch.float16,
            buffer_dtype=torch.float16,
        )

    # CPU offload config
    cpu_offload = CPUOffload(offload_params=True) if args.cpu_offload else None

    # Wrap model with FSDP
    model = FSDP(
        model,
        auto_wrap_policy=my_auto_wrap_policy,
        sharding_strategy=sharding_strategy,
        mixed_precision=mixed_precision_config,
        cpu_offload=cpu_offload,
        device_id=local_rank,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
    )

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device)
        test(rank, model, test_loader, loss_fn, epoch, device)
        scheduler.step()

    if rank == 0:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

    cleanup()


if __name__ == "__main__":
    main()

Overwriting cifar-resnet-fsdp.py


In [6]:
! torchrun \
    --nnodes=1 \
    --nproc-per-node=2 \
    cifar-resnet-fsdp.py \
      --epochs=10 \
      --batch-size=512 \
      --lr=0.1 \
      --num-workers=8 \
      --sharding-strategy=FULL_SHARD

W0421 11:29:55.278000 115 torch/distributed/run.py:852] 
W0421 11:29:55.278000 115 torch/distributed/run.py:852] *****************************************
W0421 11:29:55.278000 115 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0421 11:29:55.278000 115 torch/distributed/run.py:852] *****************************************
[INFO] Using 2 GPUs
[INFO] Sharding Strategy: FULL_SHARD
[INFO] Mixed Precision:   False
[INFO] CPU Offload:       False
[INFO] Batch per GPU:     256 | Effective batch: 512
[INFO] Data dir:          ../../datasets/data-cifar
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is goin

# Multi GPU FSDP Accelerate Setup

In [19]:
%%writefile cifar-resnet-fsdp-accelerate.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from accelerate import Accelerator
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(accelerator, model, train_loader, loss_fn, optimizer, epoch):
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        accelerator.backward(loss)
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(accelerator, model, test_loader, loss_fn, epoch):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")
    args = parser.parse_args()

    torch.manual_seed(42)

    # Accelerator reads the FSDP config from custom_config.yaml (passed via --config_file in the launch command)
    accelerator = Accelerator()

    # Divide batch size so effective batch stays constant regardless of GPU count.
    # accelerator.prepare() handles the DistributedSampler automatically,
    # but it does NOT divide the batch size — that's our responsibility.
    batch_size_per_gpu = args.batch_size // accelerator.num_processes

    if accelerator.is_main_process:
        print(f"[INFO] Distributed type: {accelerator.distributed_type}")
        print(f"[INFO] Num processes:    {accelerator.num_processes}")
        print(f"[INFO] Batch per GPU:    {batch_size_per_gpu} | Effective batch: {args.batch_size}")
        print(f"[INFO] Data dir:         {args.data_dir}")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_loader = DataLoader(train_dataset, batch_size=batch_size_per_gpu,
                              shuffle=True, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size_per_gpu * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model     = build_model()
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    # Accelerator handles FSDP sharding as defined in custom_config.yaml
    model, optimizer, train_loader, test_loader = accelerator.prepare(
        model, optimizer, train_loader, test_loader
    )

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(accelerator, model, train_loader, loss_fn, optimizer, epoch)
        test(accelerator, model, test_loader, loss_fn, epoch)
        scheduler.step()

    if accelerator.is_main_process:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")


if __name__ == "__main__":
    main()

Overwriting cifar-resnet-fsdp-accelerate.py


In [13]:
%%writefile custom_config.yaml

compute_environment: LOCAL_MACHINE
distributed_type: FSDP
num_processes: 2
num_machines: 1
machine_rank: 0
mixed_precision: fp16
downcast_bf16: no

fsdp_config:
  sharding_strategy: full_shard
  offload_params: false
  auto_wrap_policy: TRANSFORMER_BASED_WRAP
  transformer_layer_cls_to_wrap: null
  min_num_params: 1e7
  backward_prefetch_policy: BACKWARD_PRE
  forward_prefetch: false
  limit_all_gathers: true
  use_orig_params: true
  sync_module_states: true
  activation_checkpointing: false

dynamo_backend: no
use_cpu: false
debug: false

Overwriting custom_config.yaml


In [21]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from accelerate import Accelerator
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

model = build_model()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total:,} | Trainable: {trainable:,}")

Total: 21,282,122 | Trainable: 21,282,122


In [16]:
! accelerate launch \
    --dynamo_backend no \
    --config_file ./custom_config.yaml \
    cifar-resnet-fsdp-accelerate.py \
    --epochs 10 \
    --batch-size 512

[INFO] Distributed type: DistributedType.FSDP
[INFO] Num processes:    2
[INFO] Batch per GPU:    256 | Effective batch: 512
[INFO] Data dir:         ../../datasets/data-cifar
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running s